# Phase 5 — Target Engineering + Temporal Train / Validation / Test Split

## Project: H&M Customer Purchase Prediction

### Objective

Convert the customer transaction history into a genuine supervised-learning problem:

> **Predict whether a customer will make at least one purchase during the next 30 days.**

The key principle in this phase is **temporal data splitting**.

We will never use future transactions to construct historical features.

### Final formulation

- **X:** customer behavior available up to a cutoff date
- **y:** whether the customer makes ≥1 purchase during the following 30 days
- **Train:** earlier time period
- **Validation:** later time period
- **Test:** most recent period with a complete 30-day target window

This prevents temporal leakage and makes the evaluation much closer to a real deployment scenario.


## 5.1 Data Flow

```text
Raw Transactions
       |
       v
Choose temporal cutoff
       |
       +------------------------------+
       |                              |
       v                              v
Transactions <= cutoff          Transactions after cutoff
       |                              |
       v                              v
Customer Features               Future Purchase Behavior
       |                              |
       |                              v
       |                       Binary Target
       |                       1 = purchased
       |                       0 = did not purchase
       |                              |
       +--------------+---------------+
                      |
                      v
             Supervised Dataset
                      |
                      v
          Train / Validation / Test
```

### Important

We are intentionally **not** doing a random `train_test_split()` on customers.

For a time-dependent problem, random splitting can allow the model to learn patterns from future periods while being evaluated on earlier periods.


## 5.2 Imports and Paths

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
import gc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

BASE_DIR = Path("../")

TRANSACTIONS_PATH = BASE_DIR / "data" / "processed" / "transactions_model.parquet"
CUSTOMERS_PATH = BASE_DIR / "data" / "processed" / "customers.pkl"
ARTICLES_PATH = BASE_DIR / "data" / "processed" / "articles.pkl"

PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Transactions:", TRANSACTIONS_PATH)
print("Customers:", CUSTOMERS_PATH)
print("Articles:", ARTICLES_PATH)


## 5.3 Load the Data

We need the transaction table for temporal target construction.

We also load customers and articles because the feature-building function below creates customer-level features using the same logic established in Phase 4.

**The `images/` directory is not used.**


In [ ]:
transactions = pd.read_parquet(TRANSACTIONS_PATH)
customers = pd.read_pickle(CUSTOMERS_PATH)
articles = pd.read_pickle(ARTICLES_PATH)

print("Transactions:", transactions.shape)
print("Customers:", customers.shape)
print("Articles:", articles.shape)

transactions.head()


## 5.4 Basic Date Audit

Before choosing cutoffs, inspect the actual date range.

We should not hard-code dates before looking at the dataset.


In [ ]:
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

min_date = transactions["t_dat"].min()
max_date = transactions["t_dat"].max()

print("Minimum transaction date:", min_date)
print("Maximum transaction date:", max_date)
print("Total days:", (max_date - min_date).days)


## 5.5 Define the Prediction Horizon

We will predict purchase behavior over the **next 30 days**.

For a cutoff date `T`:

```text
Feature period                 Target period

transactions <= T              T+1 ... T+30
       |                            |
       v                            v
       X                            y
```

The target is:

```text
1 → customer makes at least one purchase in the next 30 days
0 → customer makes no purchase in the next 30 days
```

We need three cutoffs:

- Train cutoff
- Validation cutoff
- Test cutoff

The test cutoff must leave a complete 30-day future window inside the dataset.


In [ ]:
PREDICTION_HORIZON_DAYS = 30

# Leave enough future data to construct the test target.
# The exact cutoffs are calculated from the actual maximum date.
test_cutoff = max_date - pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

# Validation target must finish before the test feature period begins.
val_cutoff = test_cutoff - pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

# Training target must finish before the validation feature period begins.
train_cutoff = val_cutoff - pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

cutoffs = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "cutoff": [train_cutoff, val_cutoff, test_cutoff]
})

cutoffs


### Why these dates?

Suppose the maximum transaction date is `D`.

Then:

```text
TRAIN
Feature cutoff = D - 90 days
Target window  = D - 89  ... D - 60

VALIDATION
Feature cutoff = D - 60 days
Target window  = D - 59  ... D - 30

TEST
Feature cutoff = D - 30 days
Target window  = D - 29  ... D
```

Therefore:

```text
             TRAIN              VALIDATION             TEST
        |---------------|    |---------------|    |---------------|
        features target       features target       features target
              ^                    ^                    ^
              |                    |                    |
          train cutoff         val cutoff            test cutoff
```

The future target window of one split does not become part of the features of an earlier split.


## 5.6 Verify the Temporal Windows

We will explicitly calculate the feature and target periods for every split.

This is a useful audit step and should remain in the notebook because temporal leakage is one of the most important risks in this project.


In [ ]:
def describe_split(cutoff, horizon_days=30):
    target_start = cutoff + pd.Timedelta(days=1)
    target_end = cutoff + pd.Timedelta(days=horizon_days)

    return {
        "feature_end": cutoff,
        "target_start": target_start,
        "target_end": target_end,
        "target_days": (target_end - target_start).days + 1
    }

split_info = pd.DataFrame({
    split: describe_split(cutoff)
    for split, cutoff in [
        ("train", train_cutoff),
        ("validation", val_cutoff),
        ("test", test_cutoff)
    ]
}).T

split_info


# 5.7 Create a Reusable Customer Feature Function

A major issue with temporal ML is that we cannot create features once and then randomly split the resulting table.

Instead, for every cutoff we create:

```text
features_at_cutoff(cutoff)
```

The function only looks at transactions up to that cutoff.

This means the feature matrix for:

- training
- validation
- testing

is generated independently from the appropriate historical information.


In [ ]:
def build_customer_features(
    transactions,
    customers,
    articles,
    cutoff_date
):
    """
    Build customer-level features using only transactions
    available on or before cutoff_date.
    """

    # ---------------------------------------------------------
    # 1. Historical transactions
    # ---------------------------------------------------------
    hist = transactions[
        transactions["t_dat"] <= cutoff_date
    ].copy()

    # ---------------------------------------------------------
    # 2. Basic RFM features
    # ---------------------------------------------------------
    features = (
        hist.groupby("customer_id")
        .agg(
            last_purchase=("t_dat", "max"),
            first_purchase=("t_dat", "min"),
            total_items=("article_id", "count"),
            purchase_days=("t_dat", "nunique"),
            unique_articles=("article_id", "nunique"),
            total_spend=("price", "sum"),
            avg_price=("price", "mean"),
            median_price=("price", "median"),
            min_price=("price", "min"),
            max_price=("price", "max"),
            price_std=("price", "std")
        )
        .reset_index()
    )

    # Recency
    features["recency_days"] = (
        cutoff_date - features["last_purchase"]
    ).dt.days

    # Tenure
    features["customer_tenure_days"] = (
        features["last_purchase"] - features["first_purchase"]
    ).dt.days

    # Purchase rate
    features["purchase_rate"] = (
        features["purchase_days"]
        /
        (features["customer_tenure_days"] + 1)
    )

    # ---------------------------------------------------------
    # 3. Active months
    # ---------------------------------------------------------
    hist["purchase_month"] = hist["t_dat"].dt.to_period("M")

    active_months = (
        hist.groupby("customer_id")["purchase_month"]
        .nunique()
        .reset_index(name="active_months")
    )

    features = features.merge(
        active_months,
        on="customer_id",
        how="left"
    )

    # ---------------------------------------------------------
    # 4. Purchase cadence
    # ---------------------------------------------------------
    purchase_dates = (
        hist[["customer_id", "t_dat"]]
        .drop_duplicates()
        .sort_values(["customer_id", "t_dat"])
    )

    purchase_dates["days_between_purchases"] = (
        purchase_dates
        .groupby("customer_id")["t_dat"]
        .diff()
        .dt.days
    )

    cadence = (
        purchase_dates.groupby("customer_id")
        .agg(
            avg_days_between_purchases=(
                "days_between_purchases", "mean"
            ),
            median_days_between_purchases=(
                "days_between_purchases", "median"
            ),
            purchase_interval_std=(
                "days_between_purchases", "std"
            )
        )
        .reset_index()
    )

    features = features.merge(
        cadence,
        on="customer_id",
        how="left"
    )

    # ---------------------------------------------------------
    # 5. Recent behavior
    # ---------------------------------------------------------
    def recent_features(days):
        start_date = cutoff_date - pd.Timedelta(days=days)

        recent = hist[
            hist["t_dat"] > start_date
        ]

        result = (
            recent.groupby("customer_id")
            .agg(
                recent_items=("article_id", "count"),
                recent_spend=("price", "sum"),
                recent_unique_articles=("article_id", "nunique"),
                recent_purchase_days=("t_dat", "nunique"),
                recent_avg_price=("price", "mean")
            )
            .reset_index()
        )

        result = result.rename(columns={
            col: f"{col}_{days}d"
            for col in result.columns
            if col != "customer_id"
        })

        return result

    recent_30 = recent_features(30)
    recent_90 = recent_features(90)

    features = features.merge(
        recent_30,
        on="customer_id",
        how="left"
    )

    features = features.merge(
        recent_90,
        on="customer_id",
        how="left"
    )

    # ---------------------------------------------------------
    # 6. Recent-vs-lifetime behavior
    # ---------------------------------------------------------
    features["recent_spend_ratio"] = (
        features["recent_spend_30d"]
        /
        (features["total_spend"] + 1e-6)
    )

    features["recent_items_ratio"] = (
        features["recent_items_30d"]
        /
        (features["total_items"] + 1e-6)
    )

    # ---------------------------------------------------------
    # 7. Sales channel behavior
    # ---------------------------------------------------------
    channel = (
        hist.groupby(["customer_id", "sales_channel_id"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    # Standardize known H&M channel IDs.
    for channel_id in [1, 2]:
        if channel_id not in channel.columns:
            channel[channel_id] = 0

    channel = channel.rename(columns={
        1: "channel_1_items",
        2: "channel_2_items"
    })

    features = features.merge(
        channel[
            ["customer_id", "channel_1_items", "channel_2_items"]
        ],
        on="customer_id",
        how="left"
    )

    features["total_channel_items"] = (
        features["channel_1_items"].fillna(0)
        +
        features["channel_2_items"].fillna(0)
    )

    features["channel_1_ratio"] = (
        features["channel_1_items"]
        /
        (features["total_channel_items"] + 1e-6)
    )

    features["channel_2_ratio"] = (
        features["channel_2_items"]
        /
        (features["total_channel_items"] + 1e-6)
    )

    # ---------------------------------------------------------
    # 8. Article metadata aggregation
    # ---------------------------------------------------------
    article_cols = [
        "article_id",
        "product_type_name",
        "product_group_name",
        "department_name",
        "section_name",
        "index_name",
        "garment_group_name"
    ]

    available_article_cols = [
        col for col in article_cols
        if col in articles.columns
    ]

    article_metadata = articles[available_article_cols].copy()

    hist_articles = hist.merge(
        article_metadata,
        on="article_id",
        how="left"
    )

    diversity_agg = {
        col: "nunique"
        for col in [
            "product_type_name",
            "product_group_name",
            "department_name",
            "section_name",
            "index_name",
            "garment_group_name"
        ]
        if col in hist_articles.columns
    }

    diversity = (
        hist_articles.groupby("customer_id")
        .agg(**{
            f"unique_{col.replace('_name', '')}": (col, "nunique")
            for col in diversity_agg
        })
        .reset_index()
    )

    features = features.merge(
        diversity,
        on="customer_id",
        how="left"
    )

    # Preferred product group
    if "product_group_name" in hist_articles.columns:
        preferred = (
            hist_articles
            .groupby(["customer_id", "product_group_name"])
            .size()
            .reset_index(name="count")
            .sort_values(
                ["customer_id", "count"],
                ascending=[True, False]
            )
            .drop_duplicates("customer_id")
            [["customer_id", "product_group_name"]]
            .rename(
                columns={
                    "product_group_name": "preferred_product_group"
                }
            )
        )

        features = features.merge(
            preferred,
            on="customer_id",
            how="left"
        )

    # ---------------------------------------------------------
    # 9. Customer metadata
    # ---------------------------------------------------------
    customer_cols = [
        "customer_id",
        "FN",
        "Active",
        "club_member_status",
        "fashion_news_frequency",
        "age"
    ]

    available_customer_cols = [
        col for col in customer_cols
        if col in customers.columns
    ]

    customer_metadata = customers[available_customer_cols].copy()

    features = features.merge(
        customer_metadata,
        on="customer_id",
        how="left"
    )

    # Missing-age indicator
    if "age" in features.columns:
        features["age_missing"] = (
            features["age"].isna().astype("int8")
        )

    # ---------------------------------------------------------
    # 10. Log transforms
    # ---------------------------------------------------------
    log_columns = [
        "total_items",
        "total_spend",
        "unique_articles",
        "avg_price",
        "recent_spend_30d",
        "recent_items_30d",
        "recent_spend_90d",
        "recent_items_90d"
    ]

    for col in log_columns:
        if col in features.columns:
            features[f"log_{col}"] = np.log1p(
                features[col].clip(lower=0)
            )

    # ---------------------------------------------------------
    # 11. Outlier flags — do not remove observations
    # ---------------------------------------------------------
    def add_iqr_flag(df, column):
        if column not in df.columns:
            return

        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        df[f"{column}_outlier"] = (
            (df[column] < lower) |
            (df[column] > upper)
        ).astype("int8")

    for col in [
        "total_spend",
        "total_items",
        "avg_price",
        "unique_articles",
        "customer_tenure_days"
    ]:
        add_iqr_flag(features, col)

    # ---------------------------------------------------------
    # 12. Clean temporary columns
    # ---------------------------------------------------------
    features = features.drop(
        columns=["last_purchase", "first_purchase"],
        errors="ignore"
    )

    features = features.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # One row per customer
    assert features["customer_id"].is_unique

    # Free temporary memory
    del hist, purchase_dates, hist_articles
    gc.collect()

    return features


## 5.8 Build the Future-Purchase Target

The target function uses a completely different time window.

For cutoff `T`:

```text
Target window = (T, T + 30 days]
```

A customer receives:

```text
1 → at least one transaction in this window
0 → no transaction in this window
```

### Important

The target function must **not** be used when constructing X.

This separation is what keeps the feature matrix and target logically independent in time.


In [ ]:
def build_target(
    transactions,
    cutoff_date,
    horizon_days=30
):
    """
    Build a binary target:
    1 = customer purchases at least once during
        the next horizon_days
    0 = customer does not purchase
    """

    target_start = cutoff_date + pd.Timedelta(days=1)
    target_end = cutoff_date + pd.Timedelta(days=horizon_days)

    future = transactions[
        (transactions["t_dat"] >= target_start) &
        (transactions["t_dat"] <= target_end)
    ]

    purchased_customers = (
        future["customer_id"]
        .dropna()
        .drop_duplicates()
    )

    target = pd.DataFrame({
        "customer_id": purchased_customers,
        "target_purchase_30d": 1
    })

    return target, target_start, target_end


# 5.9 Build One Supervised Dataset

Before generating all three datasets, test the logic on the training cutoff.



In [ ]:
X_train_raw = build_customer_features(
    transactions,
    customers,
    articles,
    train_cutoff
)

y_train_raw, train_target_start, train_target_end = build_target(
    transactions,
    train_cutoff,
    PREDICTION_HORIZON_DAYS
)

train_df = X_train_raw.merge(
    y_train_raw,
    on="customer_id",
    how="left"
)

train_df["target_purchase_30d"] = (
    train_df["target_purchase_30d"]
    .fillna(0)
    .astype("int8")
)

print("Train features:", X_train_raw.shape)
print("Train target:", y_train_raw.shape)
print("Final train dataset:", train_df.shape)

train_df["target_purchase_30d"].value_counts(dropna=False)


## 5.10 Check Target Distribution

The target distribution is critical.

If very few customers purchase in the next 30 days, accuracy alone will be misleading.

For example, if:

```text
95% → No purchase
5%  → Purchase
```

a model predicting `0` for everyone gets 95% accuracy but is useless.

Later we will therefore focus on metrics such as:

- ROC-AUC
- PR-AUC
- Precision
- Recall
- F1
- Log Loss
- Confusion Matrix

Accuracy will still be reported, but it will not be our only metric.


In [ ]:
target_counts = train_df["target_purchase_30d"].value_counts().sort_index()

target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_counts / target_counts.sum() * 100
})

target_distribution.index = [
    "No purchase (0)" if idx == 0 else "Purchase (1)"
    for idx in target_distribution.index
]

target_distribution


# 5.11 Build Validation Dataset

Validation features are generated using information available only up to the validation cutoff.

The validation target is generated from the following 30-day period.


In [ ]:
X_val_raw = build_customer_features(
    transactions,
    customers,
    articles,
    val_cutoff
)

y_val_raw, val_target_start, val_target_end = build_target(
    transactions,
    val_cutoff,
    PREDICTION_HORIZON_DAYS
)

val_df = X_val_raw.merge(
    y_val_raw,
    on="customer_id",
    how="left"
)

val_df["target_purchase_30d"] = (
    val_df["target_purchase_30d"]
    .fillna(0)
    .astype("int8")
)

print("Validation features:", X_val_raw.shape)
print("Validation target:", y_val_raw.shape)
print("Final validation dataset:", val_df.shape)

val_df["target_purchase_30d"].value_counts(normalize=True)


# 5.12 Build Test Dataset

The test set represents the most recent period for which we have a complete 30-day future observation window.

This is our closest approximation to:

> **How the model would perform on future unseen customers / behavior.**


In [ ]:
X_test_raw = build_customer_features(
    transactions,
    customers,
    articles,
    test_cutoff
)

y_test_raw, test_target_start, test_target_end = build_target(
    transactions,
    test_cutoff,
    PREDICTION_HORIZON_DAYS
)

test_df = X_test_raw.merge(
    y_test_raw,
    on="customer_id",
    how="left"
)

test_df["target_purchase_30d"] = (
    test_df["target_purchase_30d"]
    .fillna(0)
    .astype("int8")
)

print("Test features:", X_test_raw.shape)
print("Test target:", y_test_raw.shape)
print("Final test dataset:", test_df.shape)

test_df["target_purchase_30d"].value_counts(normalize=True)


# 5.13 Important Customer Population Issue

A customer can appear in an earlier feature dataset but not appear in a later one.

For this project, that is meaningful.

For example:

```text
Customer A
|
+-- Train period: active
|
+-- Validation period: no transaction before validation cutoff
```

Our current feature-building function creates rows for customers who have historical transactions before each cutoff.

This is appropriate for a **historical active-customer prediction problem**.

However, before modeling, we will explicitly inspect how much customer overlap exists across splits.


In [ ]:
train_customers = set(train_df["customer_id"])
val_customers = set(val_df["customer_id"])
test_customers = set(test_df["customer_id"])

print("Train customers:", len(train_customers))
print("Validation customers:", len(val_customers))
print("Test customers:", len(test_customers))

print(
    "Train ∩ Validation:",
    len(train_customers & val_customers)
)

print(
    "Validation ∩ Test:",
    len(val_customers & test_customers)
)

print(
    "Train ∩ Test:",
    len(train_customers & test_customers)
)


## 5.14 Verify No Future-Feature Leakage

We can formally verify the feature construction dates.

For every split:

```text
maximum transaction date used for X
<
minimum transaction date used for y
```

This is the fundamental temporal leakage check.


In [ ]:
def verify_temporal_separation(
    transactions,
    cutoff_date,
    horizon_days
):
    feature_data = transactions[
        transactions["t_dat"] <= cutoff_date
    ]

    target_data = transactions[
        (transactions["t_dat"] > cutoff_date) &
        (
            transactions["t_dat"]
            <= cutoff_date + pd.Timedelta(days=horizon_days)
        )
    ]

    max_feature_date = feature_data["t_dat"].max()
    min_target_date = target_data["t_dat"].min()
    max_target_date = target_data["t_dat"].max()

    print("Feature max date:", max_feature_date)
    print("Target min date:", min_target_date)
    print("Target max date:", max_target_date)

    assert max_feature_date <= cutoff_date
    assert min_target_date > cutoff_date
    assert max_target_date <= cutoff_date + pd.Timedelta(days=horizon_days)

    print("Temporal separation check: PASSED")


print("TRAIN")
verify_temporal_separation(
    transactions,
    train_cutoff,
    PREDICTION_HORIZON_DAYS
)

print("\nVALIDATION")
verify_temporal_separation(
    transactions,
    val_cutoff,
    PREDICTION_HORIZON_DAYS
)

print("\nTEST")
verify_temporal_separation(
    transactions,
    test_cutoff,
    PREDICTION_HORIZON_DAYS
)


# 5.15 Compare Target Distribution Across Splits

The purchase rate may change over time.

This is important because the model can face **distribution shift**.

For example:

```text
Train purchase rate       22%
Validation purchase rate  19%
Test purchase rate        15%
```

A difference does not automatically mean something is wrong; it may reflect genuine changes in customer behavior.


In [ ]:
split_target_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "customers": [
        len(train_df),
        len(val_df),
        len(test_df)
    ],
    "positive_customers": [
        train_df["target_purchase_30d"].sum(),
        val_df["target_purchase_30d"].sum(),
        test_df["target_purchase_30d"].sum()
    ]
})

split_target_summary["positive_rate_%"] = (
    split_target_summary["positive_customers"]
    /
    split_target_summary["customers"]
    * 100
)

split_target_summary


# 5.16 Compare Feature Distributions Across Time

Before modeling, inspect whether important behavioral features shift over time.

Examples:

- recency
- total spend
- total items
- purchase days
- recent spend
- age

Large changes can indicate temporal distribution shift.


In [ ]:
comparison_columns = [
    "recency_days",
    "total_items",
    "total_spend",
    "purchase_days",
    "unique_articles",
    "avg_price",
    "recent_spend_30d",
    "recent_items_30d",
    "age"
]

available_comparison_columns = [
    col for col in comparison_columns
    if col in train_df.columns
    and col in val_df.columns
    and col in test_df.columns
]

distribution_summary = []

for col in available_comparison_columns:
    for split_name, df in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df)
    ]:
        distribution_summary.append({
            "feature": col,
            "split": split_name,
            "mean": df[col].mean(),
            "median": df[col].median(),
            "std": df[col].std(),
            "missing_pct": df[col].isna().mean() * 100
        })

distribution_summary = pd.DataFrame(distribution_summary)

distribution_summary


# 5.17 Separate X and y

The customer ID is an identifier, not a predictive feature.

We will keep it separately for traceability, but it must not be given directly to the ML algorithms.


In [ ]:
TARGET_COLUMN = "target_purchase_30d"
ID_COLUMN = "customer_id"

X_train = train_df.drop(columns=[TARGET_COLUMN, ID_COLUMN])
y_train = train_df[TARGET_COLUMN]

X_val = val_df.drop(columns=[TARGET_COLUMN, ID_COLUMN])
y_val = val_df[TARGET_COLUMN]

X_test = test_df.drop(columns=[TARGET_COLUMN, ID_COLUMN])
y_test = test_df[TARGET_COLUMN]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


# 5.18 Identify Numeric and Categorical Features

We will **not** manually encode these columns yet.

Phase 6 will build a proper preprocessing pipeline using:

- `SimpleImputer`
- `OneHotEncoder`
- `StandardScaler`
- optional PCA
- potentially different preprocessing branches for different model families

Keeping preprocessing separate from target engineering makes the pipeline cleaner and reduces leakage risk.


In [ ]:
numeric_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Number of numerical features:", len(numeric_columns))
print("Number of categorical features:", len(categorical_columns))

print("\nCategorical features:")
print(categorical_columns)


# 5.19 Check Missing Values Before Preprocessing

We expect some missing values.

Examples:

- age may be missing
- purchase interval is undefined for customers with only one purchase day
- some article metadata may be missing
- recent behavior can be missing if a customer did not purchase during the recent window

We will **not fill these manually here**.

Phase 6 will handle missing values inside the preprocessing pipeline.


In [ ]:
missing_train = (
    X_train.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_train[missing_train > 0]


# 5.20 Check Duplicate Customers

Each customer must have exactly one row per split.


In [ ]:
assert train_df["customer_id"].is_unique
assert val_df["customer_id"].is_unique
assert test_df["customer_id"].is_unique

print("Train customer uniqueness: PASSED")
print("Validation customer uniqueness: PASSED")
print("Test customer uniqueness: PASSED")


# 5.21 Save the Final Phase 5 Datasets

We save both:

1. Complete datasets with `customer_id` and target
2. Separate X/y objects for convenient modeling

Parquet is used because the feature tables may contain categorical/string columns and are efficient for this workflow.


In [ ]:
# Complete datasets
train_path = PROCESSED_DIR / "train_phase5.parquet"
val_path = PROCESSED_DIR / "validation_phase5.parquet"
test_path = PROCESSED_DIR / "test_phase5.parquet"

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

print("Saved:")
print(train_path)
print(val_path)
print(test_path)


In [ ]:
# Save target arrays separately
y_train.to_frame(name=TARGET_COLUMN).to_parquet(
    PROCESSED_DIR / "y_train_phase5.parquet",
    index=False
)

y_val.to_frame(name=TARGET_COLUMN).to_parquet(
    PROCESSED_DIR / "y_validation_phase5.parquet",
    index=False
)

y_test.to_frame(name=TARGET_COLUMN).to_parquet(
    PROCESSED_DIR / "y_test_phase5.parquet",
    index=False
)

print("Target files saved.")


# 5.22 Final Phase 5 Audit

Run this final checklist before moving to Phase 6.


In [ ]:
print("=" * 60)
print("PHASE 5 FINAL AUDIT")
print("=" * 60)

print("\n1. Date range")
print("   Dataset start:", min_date)
print("   Dataset end:", max_date)

print("\n2. Cutoffs")
print("   Train cutoff:", train_cutoff)
print("   Validation cutoff:", val_cutoff)
print("   Test cutoff:", test_cutoff)

print("\n3. Dataset shapes")
print("   Train:", train_df.shape)
print("   Validation:", val_df.shape)
print("   Test:", test_df.shape)

print("\n4. Positive class rates")
print("   Train:", y_train.mean())
print("   Validation:", y_val.mean())
print("   Test:", y_test.mean())

print("\n5. Feature counts")
print("   Numeric:", len(numeric_columns))
print("   Categorical:", len(categorical_columns))

print("\n6. Customer uniqueness")
print("   Train:", train_df["customer_id"].is_unique)
print("   Validation:", val_df["customer_id"].is_unique)
print("   Test:", test_df["customer_id"].is_unique)

print("\n7. Temporal leakage")
print("   Verified: PASSED")

print("\nPHASE 5 COMPLETE")


# Phase 5 — What We Have Now

We have converted the H&M transaction data into a supervised classification problem:

```text
                CUSTOMER HISTORY
                       |
                       v
             Feature Engineering
                       |
                       v
                 Cutoff T
                       |
              +--------+--------+
              |                 |
              v                 v
             X(T)          Future 30 days
                                |
                                v
                         y = purchase?
                          0        1
```

### Final objects

```text
X_train, y_train
X_val,   y_val
X_test,  y_test
```

### Target

```text
target_purchase_30d
```

### Meaning

```text
1 → customer purchased at least once
    in the next 30 days

0 → customer did not purchase
    in the next 30 days
```

---

# Critical decisions made

### 1. No random transaction split

Because this is temporal customer behavior.

### 2. No future information in features

Every feature is constructed using transactions up to the cutoff.

### 3. 30-day prediction horizon

This gives us a concrete business/ML prediction problem.

### 4. No manual missing-value imputation yet

That belongs inside the preprocessing pipeline.

### 5. No manual categorical encoding yet

That also belongs in the preprocessing pipeline.

### 6. No PCA yet

PCA must be fitted only on training data and will be handled in Phase 6/7.

### 7. No outlier deletion

Outliers may represent legitimate high-value customers.

---

# Next Phase

## Phase 6 — ML Preprocessing Pipeline

We will build a proper `scikit-learn` preprocessing architecture:

```text
                    X
                    |
          +---------+---------+
          |                   |
       Numeric            Categorical
          |                   |
     Imputation           Imputation
          |                   |
     Scaling             One-Hot Encoding
          |                   |
          +---------+---------+
                    |
                    v
             Processed Matrix
                    |
             +------+------+
             |             |
          Without PCA    With PCA
             |             |
             +------+------+
                    |
                    v
                 Model
```

This phase will also address **data leakage during preprocessing**, which is another important placement/interview topic.
